# NER for Sentences

### Authors - Shivendra Raghav (1001123853), Amirhosein Morteza (1002187551)
### CSE 5388-002

#### This file trains the NER model for the sentences.

#### References: https://spacy.io/usage/training

### Import the required libraries

In [1]:
import spacy
import random
import pandas as pd
from spacy.scorer import Scorer
from spacy.training import Example
from spacy.util import minibatch, compounding
from sklearn.model_selection import train_test_split

### Processing the NER metadata

In [6]:
# Reading the NER Sentence Dataset
df = pd.read_csv("../synthetic_dataset/sentence_classified_data_ner.csv")

# Filter the sensitive sentences
df_sensitive = df[df["sensitive"] == 1]

# Extract the data from the CSV file
ner_metadata = []
for _, row in df_sensitive.iterrows():
    entities = eval(row["ner_metadata"])
    ner_metadata.append((row["sentence"], {"entities": entities}))

ner_metadata[0]

('Please find the access details for the ISDA Master Agreement Database below:\n\nUsername: priyansh45\nAccess Key: 781959-396g0bqx-guak-vlk8-ok8k-4jw8cbxbuu9b',
 {'entities': [(88, 98, 'USERNAME')]})

### Split the data into training and testing and create the NER model

In [8]:
# Split the data into training and testing sets
train_data, test_data = train_test_split(ner_metadata, test_size=0.2, random_state=42)

# Load the spaCy model and create a blank EntityRecognizer
nlp = spacy.blank("en")
ner = nlp.add_pipe("ner", last=True)

# Adding our labels to the EntityRecognizer
for _, annotations in train_data:
    for entity in annotations.get("entities"):
        ner.add_label(entity[2])

ner.labels

('API_KEY', 'DATABASE_CONN', 'PASSWORD', 'SSN', 'USERNAME')

### SpaCy NER model

In [9]:
train_examples = []
for text, annotations in train_data:
    doc = nlp.make_doc(text)
    example = Example.from_dict(doc, annotations)
    train_examples.append(example)

optimizer = nlp.initialize()
for itn in range(20):
    random.shuffle(train_examples)
    losses = {}
    batches = minibatch(train_examples, size=compounding(4.0, 32.0, 1.001))
    for batch in batches:
        nlp.update(batch, sgd=optimizer, drop=0.3, losses=losses)
    print("Iteration:", itn, "Losses:", losses)

Iteration: 0 Losses: {'ner': 3073.4628916712745}
Iteration: 1 Losses: {'ner': 1349.9863490280773}
Iteration: 2 Losses: {'ner': 916.6835422483482}
Iteration: 3 Losses: {'ner': 660.197669627574}
Iteration: 4 Losses: {'ner': 402.17845020343213}
Iteration: 5 Losses: {'ner': 338.5956483968384}
Iteration: 6 Losses: {'ner': 266.30829778435805}
Iteration: 7 Losses: {'ner': 198.44941988849044}
Iteration: 8 Losses: {'ner': 182.45015460447607}
Iteration: 9 Losses: {'ner': 181.5366906007423}
Iteration: 10 Losses: {'ner': 139.11355394835758}
Iteration: 11 Losses: {'ner': 117.24619538879054}
Iteration: 12 Losses: {'ner': 81.3674779934928}
Iteration: 13 Losses: {'ner': 96.26347213065402}
Iteration: 14 Losses: {'ner': 113.97301046304332}
Iteration: 15 Losses: {'ner': 94.25398214199222}
Iteration: 16 Losses: {'ner': 92.46491549462165}
Iteration: 17 Losses: {'ner': 84.50995288919846}
Iteration: 18 Losses: {'ner': 84.56874613794363}
Iteration: 19 Losses: {'ner': 68.49653221935658}


### Evalutation of the NER model 

In [12]:
# Evaluate the trained model using Scorer
scorer = Scorer(nlp)
examples = []
for text, annotations in test_data:
    doc = nlp(text)
    example = Example.from_dict(doc, annotations)
    scorer.score([example])
    examples.append(example)

# Retrieve and print the evaluation metrics
scores = scorer.score(examples)

df_scores = pd.DataFrame(columns=["Metric", "Value"])
df_scores = df_scores._append({"Metric": "Precision", "Value":scores['ents_p']}, ignore_index=True)
df_scores = df_scores._append({"Metric": "Recall", "Value":scores['ents_r']}, ignore_index=True)
df_scores = df_scores._append({"Metric": "F1 score:", "Value":scores['ents_f']}, ignore_index=True)

df_scores.head()

/var/folders/rz/4djzw0r155g8_kdwpx3gm7y40000gn/T/ipykernel_71007/1716324110.py:14: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_scores = df_scores._append({"Metric": "Precision", "Value":scores['ents_p']}, ignore_index=True)


,Metric,Value
0,Precision,0.863158
1,Recall,0.658635
2,F1 score:,0.747153


In [13]:
# Entity-level evaluation
entities_table = []
for text, annotations in test_data:
    doc = nlp(text)
    predicted_ents = [(ent.text, ent.label_) for ent in doc.ents]
    actual_ents = [(text[ent[0]:ent[1]], ent[2]) for ent in annotations['entities']]
    entities_table.extend([(pe[0], pe[1], ae[1]) for pe, ae in zip(predicted_ents, actual_ents)])

# Creating a DataFrame from the entities_table list
df_results = pd.DataFrame(entities_table, columns=["Entity", "Predicted as", "Actual type"])

df_results.head(10)

,Entity,Predicted as,Actual type
0,cc198ef2d97061138fb66abe43126905b42d923b,API_KEY,API_KEY
1,stream,PASSWORD,PASSWORD
2,download,PASSWORD,PASSWORD
3,578-47-8913,SSN,SSN
4,r5GSTEd0TLrYrLlQJ24Ijw,API_KEY,API_KEY
5,williams,PASSWORD,USERNAME
6,steffi44,USERNAME,USERNAME
7,support,PASSWORD,PASSWORD
8,herman,PASSWORD,PASSWORD
9,mysql://dev_admin:dev_s3cr3tp@ss@dev-mysql-123...,DATABASE_CONN,DATABASE_CONN
